# Field Statistics for Cleaning

In [ ]:
from pathlib import Path
import polars as pl
import ftfy
from codecarbon import EmissionsTracker
from mds_data_model.introspection import date_fields, reference_number_fields, measurement_fields, monetary_fields

In [5]:
def _flatten_verbose(pat: str) -> str:
    """Flatten a re.VERBOSE pattern so polars can use it natively"""
    out, in_class, esc, in_comment = [], False, False, False
    for ch in pat:
        if in_comment:
            if ch == "\n":
                in_comment = False
        elif esc:
            out.append(ch); esc = False
        elif ch == "\\":
            out.append(ch); esc = True
        elif in_class:
            out.append(ch)
            if ch == "]":
                in_class = False
        elif ch == "[":
            in_class = True; out.append(ch)
        elif ch == "#":
            in_comment = True
        elif not ch.isspace():
            out.append(ch)
    return "".join(out)


DATA_PATH = Path("../data/mds-flat-records.parquet")

INTERMEDIATE_PATH = Path("analysis_output")
INTERMEDIATE_PATH.mkdir(parents=True, exist_ok=True)

# set up directory for codecarbon logs
EMISSIONS_LOG_PATH = INTERMEDIATE_PATH / "emissions_logs"
EMISSIONS_LOG_PATH.mkdir(parents=True, exist_ok=True)


STOPWORDS = frozenset({
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your",
    "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she",
    "her", "hers", "herself", "it", "its", "itself", "they", "them", "their",
    "theirs", "themselves", "what", "which", "who", "whom", "this", "that",
    "these", "those", "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an",
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of",
    "at", "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down",
    "in", "out", "on", "off", "over", "under", "again", "further", "then",
    "once", "here", "there", "when", "where", "why", "how", "all", "any",
    "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor",
    "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can",
    "will", "just", "don", "should", "now",
})
STOPWORD_REGEX = r"(?i)\b(?:" + "|".join(sorted(STOPWORDS)) + r")\b"

_TLD = r"com|org|net|edu|gov|mil|int|io|co|ai|app|dev|info|biz|xyz|uk|us|ca|de|fr|jp|cn|au|in|ru|nl|eu"
URL_REGEX = (
    r"(?i)\b(?:https?://)?"
    r"(?:[a-z0-9](?:[a-z0-9-]*[a-z0-9])?\.)+"
    rf"(?:{_TLD})\b(?:/[^\s]*)?"
)

EMAIL_REGEX = (
    r"(?i)\b[a-z0-9._%+-]+"
    r"@[a-z0-9](?:[a-z0-9-]*[a-z0-9])?"
    r"(?:\.[a-z0-9](?:[a-z0-9-]*[a-z0-9])?)*"
    r"\.[a-z]{2,}\b"
)

# reformat ftfy's pattern for the rust regex engine
MOJIBAKE_REGEX = _flatten_verbose(ftfy.badness.BADNESS_RE.pattern)

ldf = pl.scan_parquet(DATA_PATH)
field_types = ldf.select(pl.col("field_type").unique()).collect(engine="streaming")["field_type"].to_list()
data_sources = ldf.select(pl.col("data_source").unique()).collect(engine="streaming")["data_source"].to_list()
base_fields = ldf.collect_schema()

In [6]:
DATE_FIELDS = pl.col("field_type").is_in(date_fields())

OBJECT_NUMBER_FIELDS = [
    "spectrum/object_number",
    "spectrum/other_number",
    "spectrum/related_object_number",
    "spectrum/disposal_new_object_number",
    "spectrum/catalogue_number",
]
REFERENCE_NUMBER_FIELDS = list(reference_number_fields())
# Distinct schemes / authority codes
OTHER_IDENTIFIER_FIELDS = [
    "spectrum/entry_number",
    "spectrum/field_collection_number",
    "spectrum/reproduction_number",
    "spectrum/transfer_of_title_number",
    "spectrum/location_reference_name_number",
    "spectrum/organisations_mda_code",
]
IDENTIFIER_FIELDS = pl.col("field_type").is_in(
    OBJECT_NUMBER_FIELDS + REFERENCE_NUMBER_FIELDS + OTHER_IDENTIFIER_FIELDS)

# Counts / edition arithmetic ("3 of 50")
COUNT_FIELDS = pl.col("field_type").is_in([
    "spectrum/number_of_objects",
    "spectrum/copy_number",
    "spectrum/edition_number",
])
PRICE_FIELDS = list(monetary_fields())
MEASUREMENT_FIELDS = list(measurement_fields())
STRUCTURED_FIELDS = pl.col("field_type").is_in(
    PRICE_FIELDS + MEASUREMENT_FIELDS)

PATTERN_FIELDS = DATE_FIELDS | STRUCTURED_FIELDS # | IDENTIFIER_FIELDS | COUNT_FIELDS # Identifiers and counts are low value and are causing lots of noise

In [7]:
def fix_mojibake(s: pl.Series) -> pl.Series:
    return pl.Series([ftfy.fix_text(v) for v in s], dtype=pl.String())

In [8]:
# Apply minimal universal standardisation
base = (
    pl.scan_parquet(DATA_PATH)
    .with_columns(
        pl.col("data_source").cast(pl.Enum(data_sources))) # Set data source as an enum to reduce memory
    .filter(
        pl.col("field_type").str.starts_with("spectrum/")
        & pl.col("value").is_not_null())
    .with_columns(
        pl.col("value").str.strip_chars())
    .filter(pl.col("value") != "")
    .with_columns(
        contains_mojibake=pl.col("value").str.contains(MOJIBAKE_REGEX))
)

# Fix mojibake before calculating other stats
fixed_mojibake = (
    base
    .filter(pl.col("contains_mojibake"))
    .select(
        pl.col("node_id"),
        pl.col("value")
        .map_batches(
            fix_mojibake, return_dtype=pl.String(), is_elementwise=True)
        .alias("value_fixed"))
)

ldf = (
    base
    .join(fixed_mojibake, on="node_id", how="left")
    .with_columns(
        pl.coalesce("value_fixed", "value").alias("value"))
    .drop("value_fixed")
    .with_columns(
        # Character-class composition
        char_count=pl.col("value").str.len_chars(),
        digit_chars=pl.col("value").str.count_matches(r"\d"),
        alpha_chars=pl.col("value").str.count_matches(r"[A-Za-z]"),
        punct_chars=pl.col("value").str.count_matches(r"[^\w\s]"),

        # Structural/delimiter hints - candidates
        delim_chars=pl.col("value").str.count_matches(r"[,;/|]"),
        paren_count=pl.col("value").str.count_matches(r"\("),

        # Linguistic units
        token_count=pl.col("value").str.count_matches(r"\b\w+\b"),
        stop_count=pl.col("value").str.count_matches(STOPWORD_REGEX),

        # Europeana contamination: a closing tag is rarely incidental
        contains_html=pl.col("value").str.contains(r"</[a-zA-Z][^>]*>"),
        contains_url=pl.col("value").str.contains(URL_REGEX),
        contains_email=pl.col("value").str.contains(EMAIL_REGEX),
        escape_count=pl.col("value").str.count_matches(r"[\n\t\r]"),

        # Case signature — distinguishes identifiers/enums from prose
        no_lowercase=pl.col("value").str.contains(r"^[^a-z]*$")))

## Pattern Induction over Structured Fields and Identifiers

1. Mask all upper and lower case alpha characters with `s` and digits with `d`
2. Run length encode masks
3. Merge all masks within a column (group) to form length ranges e.g. `d{1,3}`
    - For dates, an additional constraint is that their widest slot position must also match (to prevent conflation of Y.M.D -> D.M.Y)

In [9]:
# Shared induce_lookup over structured fields, sinking field_stats
from mds_norm.pipeline.consistency_induction import induce_lookup

with EmissionsTracker(project_name="tier0_pattern_induction", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    group_expr = (
        pl.when(pl.col("field_type").str.contains("date"))
        .then(pl.lit("__date__"))
        .when(pl.col("field_type").is_in(OBJECT_NUMBER_FIELDS))
        .then(pl.lit("__object_number__"))
        .when(pl.col("field_type").is_in(REFERENCE_NUMBER_FIELDS))
        .then(pl.lit("__reference_number__"))
        .when(pl.col("field_type").is_in(PRICE_FIELDS))
        .then(pl.lit("__price__"))
        .otherwise(pl.col("field_type"))
        .alias("merge_group"))

    scoped = (
        ldf.filter(PATTERN_FIELDS)
        .filter(pl.col("value").is_not_null())
        .select(group_expr, "value"))

    # (merge_group, value) -> pattern, slot_values, merged_pattern
    lookup = induce_lookup(scoped, keep_features=True)

    # Sink stats to disk
    base = ldf.with_columns(group_expr)
    x = (
        pl.concat(
            [base.filter(PATTERN_FIELDS).join(lookup.lazy(), on=["merge_group", "value"], how="left"),
             base.filter(~PATTERN_FIELDS)],
            how="diagonal_relaxed")
        .sink_parquet(INTERMEDIATE_PATH / "field_stats.parquet"))

[codecarbon WARNING @ 13:01:40] Multiple instances of codecarbon are allowed to run at the same time.
